# Prática 24/11 · Da previsão à decisão: perfis de municípios e carteira sob orçamento
**Aprendizagem Computacional · 4º TIA · Aula 05 · versão INICIAL**

Em 17/11, escolhemos a regressão linear pela validação cruzada e abrimos o teste uma vez (MAE de 1,23 p.p.). Hoje usamos o modelo para **decidir**, que é o que o grupo mostra no check-in de 24/11:

1. prever a abstenção de **todos** os municípios, sem otimismo, e exportar `previsoes.csv`;
2. descrever o território com **perfis** (PCA e K-Means), sem usar y;
3. escolher onde investir uma ação de mobilização com orçamento limitado, comparando a **solução ótima** (problema da mochila) com regras simples.

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | Previsões fora da amostra com `cross_val_predict` | cap. 24 |
| 2 | Padronizar e projetar com PCA em 2 componentes | cap. 6 e 7 |
| 3 | Cotovelo e silhueta para escolher o k | cap. 19 e 20 |
| 4 | Benefício previsto de cada município | cap. 24 |
| 5 | Comparar a mochila com "maior % prevista primeiro" | cap. 24 |

## 1 · Carregar a base e o modelo escolhido em 17/11 (pronto)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
pd.set_option('display.precision', 2)

ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-24_Aula05_Perfis_e_Decisao/dados/' + ARQUIVO
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL, '|', base.shape)

In [ ]:
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

Y_COL = 'ABSTENCAO_PCT'
X_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
          'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
X, y = base[X_COLS], base[Y_COL]
cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
modelo = Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())])

# Arquivos gerados ficam FORA da pasta da prática: /content no Colab, sua pasta de usuário no laboratório
SAIDA = (Path('/content') if Path('/content').exists() else Path.home() / 'ac_saida') / 'aula05'
SAIDA.mkdir(parents=True, exist_ok=True)
print('Arquivos gerados vão para:', SAIDA)

## 2 · TODO 1: uma previsão para cada município, sem otimismo (cap. 24)
Para decidir, precisamos de uma previsão para **todos** os 141 municípios. Se o modelo prever os municípios que usou no ajuste, o erro parece menor do que é.

Com `cross_val_predict(modelo, X, y, cv=cv)`, cada município é previsto por um modelo ajustado nas **outras** 4 dobras, que nunca o viu. Complete e rode: a célula também salva `previsoes.csv`, a fonte do dashboard ou do Power BI.

In [ ]:
from sklearn.model_selection import cross_val_predict

base['ABST_PREVISTA'] = None  # TODO 1: cross_val_predict(modelo, X, y, cv=cv)
base['RESIDUO'] = base[Y_COL] - base['ABST_PREVISTA']
print(f"MAE fora da amostra ({len(base)} municípios): {mean_absolute_error(y, base['ABST_PREVISTA']):.2f} p.p.")
COLS_SAIDA = ['CD_MUNICIPIO', 'NM_MUNICIPIO', 'QT_APTOS', 'DIST_CUIABA_KM', Y_COL, 'ABST_PREVISTA', 'RESIDUO']
base[COLS_SAIDA].to_csv(SAIDA / 'previsoes.csv', index=False, encoding='utf-8-sig')
base.nlargest(5, 'RESIDUO')[['NM_MUNICIPIO', Y_COL, 'ABST_PREVISTA', 'RESIDUO']]

**Pergunta para discussão:** Serra Nova Dourada teve cerca de 7 p.p. de abstenção acima do previsto. O que o gestor deve fazer com essa informação?

**Resposta esperada:** investigar antes de concluir. Um resíduo grande diz que algo nesse município não está nas variáveis do modelo: uma seção que mudou de lugar, uma estrada interditada, migração recente. O modelo aponta **onde olhar**; a explicação vem de quem conhece o lugar.

## 3 · TODO 2: o território em 2 dimensões, com PCA (cap. 6 e 7)
Os perfis descrevem os municípios **sem usar y** nem a abstenção de 2022: só perfil do eleitorado e geografia. São 6 variáveis em escalas diferentes, então primeiro padronizamos (cada uma vira "desvios-padrão da média").

A PCA resume as 6 em 2 eixos que guardam o máximo da variação, para podermos **ver** os municípios num gráfico. Complete:
- `Z = StandardScaler().fit_transform(base[PERFIL_COLS])`
- `pca = PCA(n_components=2, random_state=SEMENTE).fit(Z)` e `componentes = pca.transform(Z)`

In [ ]:
from sklearn.decomposition import PCA

PERFIL_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
               'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
Z = None            # TODO 2: padronize base[PERFIL_COLS] com StandardScaler().fit_transform
pca = None          # TODO 2: PCA(n_components=2, random_state=SEMENTE).fit(Z)
componentes = None  # TODO 2: pca.transform(Z)
print('variância explicada pelos 2 eixos:', pca.explained_variance_ratio_.round(2), '| total', round(pca.explained_variance_ratio_.sum(), 2))
pd.DataFrame(pca.components_.T, index=PERFIL_COLS, columns=['eixo 1', 'eixo 2']).round(2)

## 4 · TODO 3: quantos perfis? Cotovelo e silhueta (cap. 19 e 20)
O K-Means precisa que a gente diga o k (número de grupos). Duas pistas:
- **cotovelo:** a inércia (distância dos municípios ao centro do seu grupo) sempre cai com mais grupos; procuramos onde ela para de cair rápido;
- **silhueta:** de −1 a 1, mede se cada município está mais perto do seu grupo do que do vizinho. Quanto maior, melhor.

Dentro do `for`, ajuste `KMeans(n_clusters=k, n_init=10, random_state=SEMENTE)` em `Z` e guarde a inércia (`km.inertia_`) e a silhueta (`silhouette_score(Z, km.labels_)`).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 7)
inercia, silhueta = {}, {}
for k in ks:
    km = None              # TODO 3: KMeans(n_clusters=k, n_init=10, random_state=SEMENTE) ajustado em Z
    inercia[k] = None      # TODO 3: km.inertia_
    silhueta[k] = None     # TODO 3: silhouette_score(Z, km.labels_)
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.2))
eixos[0].plot(list(ks), list(inercia.values()), marker='o'); eixos[0].set_title('Cotovelo (inércia)'); eixos[0].set_xlabel('k')
eixos[1].plot(list(ks), list(silhueta.values()), marker='o', color='#bf3989'); eixos[1].set_title('Silhueta'); eixos[1].set_xlabel('k')
plt.tight_layout(); plt.show()
K = max(silhueta, key=silhueta.get)
print('silhuetas:', {k: round(v, 3) for k, v in silhueta.items()}, '→ k =', K)

## 5 · Os perfis (pronto)
Ajustamos o K-Means com o k escolhido, descrevemos cada grupo pela média das variáveis e só **depois** olhamos a abstenção de cada grupo. Os nomes são leitura humana da tabela.

In [ ]:
base['PERFIL'] = KMeans(n_clusters=K, n_init=10, random_state=SEMENTE).fit_predict(Z)
resumo = base.groupby('PERFIL').agg(n=('NM_MUNICIPIO', 'size'), pct_16_24=('PCT_16_24', 'mean'),
    pct_65=('PCT_65_MAIS', 'mean'), pct_sup=('PCT_SUPERIOR_COMPLETO', 'mean'),
    eleitorado_mediano=('QT_APTOS', 'median'), cresc=('CRESC_ELEITORADO_PCT', 'mean'),
    dist_km=('DIST_CUIABA_KM', 'mean'), abst_media=(Y_COL, 'mean'))
def nomear(l):
    if l['eleitorado_mediano'] == resumo['eleitorado_mediano'].max():
        return 'Polos maiores e mais escolarizados'
    if l['pct_65'] == resumo['pct_65'].max():
        return 'Pequenos, envelhecidos e perto da capital'
    return 'Longe da capital e em expansão'
resumo['nome'] = [nomear(l) for _, l in resumo.iterrows()]
base['PERFIL_NOME'] = base['PERFIL'].map(resumo['nome'])
fig, ax = plt.subplots(figsize=(7, 4.5))
for nome, grupo in base.groupby('PERFIL_NOME'):
    ax.scatter(componentes[grupo.index, 0], componentes[grupo.index, 1], s=18, label=nome, alpha=0.75)
ax.set_xlabel('eixo 1 da PCA'); ax.set_ylabel('eixo 2 da PCA'); ax.legend(fontsize=8); ax.set_title('Perfis de municípios')
plt.tight_layout(); plt.show()
resumo

**Pergunta para discussão:** o perfil "Longe da capital e em expansão" tem a maior abstenção média. O K-Means "previu" a abstenção?

**Resposta esperada:** não. Ele nunca viu a abstenção; ela foi calculada depois, por grupo, só para descrever. O que o agrupamento mostra é que distância e crescimento do eleitorado andam juntos no território, e que é nesse tipo de município que a abstenção se concentra. A silhueta baixa (perto de 0,24) também avisa: os grupos se sobrepõem. São uma descrição útil, não tipos naturais.

## 6 · Da previsão à decisão: a carteira sob orçamento (cap. 24)
**Hipóteses didáticas (declaradas, não oficiais):**
- **Meta:** abstenção de até 20%. A ação mira só o **excesso** previsto acima da meta.
- **Efeito:** a ação recupera 30% dos eleitores em excesso.
- **Custo:** R$ 10 mil fixos + R$ 1,50 por eleitor apto.

Para cada município *i*: benefício previsto bᵢ = 0,30 × max(abst_previstaᵢ − 20, 0) / 100 × aptosᵢ (eleitores a mais) e custo cᵢ = 10 + 0,0015 × aptosᵢ (R$ mil).

**TODO 4:** complete o benefício previsto. Use `np.clip(base['ABST_PREVISTA'] - META, 0, None)` para o "max(…, 0)". A coluna `BENEF_REAL`, pronta, aplica a mesma regra à abstenção que realmente ocorreu; ela só serve para a checagem retrospectiva.

In [ ]:
META, EFEITO, FIXO, POR_ELEITOR = 20.0, 0.30, 10, 0.0015
ORCAMENTO = 1000  # R$ mil

base['CUSTO'] = np.ceil(FIXO + POR_ELEITOR * base['QT_APTOS']).astype(int)
base['BENEF_PREV'] = None  # TODO 4: EFEITO × max(ABST_PREVISTA − META, 0) / 100 × QT_APTOS
base['BENEF_REAL'] = EFEITO * np.clip(base[Y_COL] - META, 0, None) / 100 * base['QT_APTOS']
print('municípios com benefício previsto > 0:', (base['BENEF_PREV'] > 0).sum(), 'de', len(base))

## 7 · TODO 5: a mochila contra as regras simples
**Problema da mochila 0/1:** escolher xᵢ ∈ {0, 1} para maximizar Σ bᵢ·xᵢ sujeito a Σ cᵢ·xᵢ ≤ orçamento. A função `mochila` (pronta) resolve por programação dinâmica e dá a solução **ótima**; a `guloso` segue uma ordem e vai escolhendo enquanto couber.

Complete a estratégia "Maior % prevista primeiro": `guloso(base['ABST_PREVISTA'].sort_values(ascending=False).index, ORCAMENTO)`.

In [ ]:
def mochila(custos, beneficios, B):
    custos, beneficios = np.asarray(custos), np.asarray(beneficios)
    dp = np.zeros(B + 1); guarda = np.zeros((len(custos), B + 1), dtype=bool)
    for i, (c, b) in enumerate(zip(custos, beneficios)):
        if b <= 0 or c > B:
            continue
        candidato = dp[:B + 1 - c] + b
        melhora = candidato > dp[c:]
        novo = dp.copy(); novo[c:][melhora] = candidato[melhora]
        guarda[i, c:] = melhora; dp = novo
    escolhidos, w = [], B
    for i in range(len(custos) - 1, -1, -1):
        if guarda[i, w]:
            escolhidos.append(i); w -= custos[i]
    return sorted(escolhidos)

def guloso(ordem, B):
    escolhidos, total = [], 0
    for i in ordem:
        if base.loc[i, 'BENEF_PREV'] > 0 and total + base.loc[i, 'CUSTO'] <= B:
            escolhidos.append(i); total += base.loc[i, 'CUSTO']
    return escolhidos

estrategias = {
    'Ótima (mochila)': mochila(base['CUSTO'], base['BENEF_PREV'], ORCAMENTO),
    'Maior % prevista primeiro': None,   # TODO 5: guloso(...) na ordem da abstenção prevista, da maior para a menor
    'Maiores municípios primeiro': guloso(base['QT_APTOS'].sort_values(ascending=False).index, ORCAMENTO),
}
tabela = pd.DataFrame([{
    'estratégia': n, 'municípios': len(s), 'custo (R$ mil)': base.loc[s, 'CUSTO'].sum(),
    'eleitores a mais (previsto)': base.loc[s, 'BENEF_PREV'].sum(),
    'eleitores a mais (retrospectivo)': base.loc[s, 'BENEF_REAL'].sum()} for n, s in estrategias.items()]).round(0)
tabela

**Pergunta para discussão:** a mochila é "ótima". Então ela sempre traz mais eleitores na vida real?

**Resposta esperada:** não. Ela é ótima **para os números que recebe**, que são previsões. Se o modelo errar justamente nos municípios que ela escolhe, a carteira inteira sofre: a otimização amplifica o erro da previsão. A coluna retrospectiva mostra que, com este modelo (erro de cerca de 1,3 p.p.), a mochila ainda vence. Com o modelo antigo do molde, que errava mais, as regras simples ganhavam dela.

In [ ]:
carteira = base.loc[estrategias['Ótima (mochila)']].sort_values('BENEF_PREV', ascending=False)
carteira[['NM_MUNICIPIO', 'PERFIL_NOME', 'QT_APTOS', 'ABST_PREVISTA', 'CUSTO', 'BENEF_PREV']].head(10)

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('previsões fora da amostra: MAE 1,28', lambda: round(mean_absolute_error(y, base['ABST_PREVISTA']), 2) == 1.28),
    conferir('previsoes.csv salvo com 141 linhas', lambda: len(pd.read_csv(SAIDA / 'previsoes.csv')) == 141),
    conferir('PCA: 2 eixos explicam 0,67 da variação', lambda: round(pca.explained_variance_ratio_.sum(), 2) == 0.67),
    conferir('silhueta escolheu k = 3', lambda: K == 3),
    conferir('benefício previsto em 130 municípios', lambda: (base['BENEF_PREV'] > 0).sum() == 130),
    conferir('mochila: 29 municípios dentro do orçamento', lambda: len(estrategias['Ótima (mochila)']) == 29 and base.loc[estrategias['Ótima (mochila)'], 'CUSTO'].sum() <= ORCAMENTO),
    conferir('"maior % prevista primeiro": 37 municípios', lambda: len(estrategias['Maior % prevista primeiro']) == 37),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as perguntas da banca, para a decisão)
1. **De onde vêm as previsões?** `cross_val_predict`: cada município previsto por um modelo que não o viu. 2. **Os perfis usam y?** Não; a abstenção só descreve os grupos depois.
3. **Quais hipóteses a decisão assume?** Meta, efeito e custo didáticos, declarados. 4. **A decisão é robusta?** Checagem retrospectiva contra regras simples.
**O que falta (01/12):** auditar tudo no Tribunal do Pipeline e garantir que roda do zero.